In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow as pa

_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

In [ ]:
def _add_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Temperature-based features from Open-Meteo historical data.

    Temperature is a top-3 driver of NSW electricity demand:
      - Summer heatwaves -> AC load surge -> demand/price spikes
      - Winter cold snaps -> heating load -> evening peak amplification
      - Solar output is correlated with warm, clear days

    Two forward-looking signals are included:
      - temp_t24: actual temperature 24h ahead (proxy for the real BOM
        24h-ahead forecast; at inference time replace with actual forecast)
      - hdd_t24 / cdd_t24: degree features at forecast horizon

    No-op if temp_sydney is absent or all-NaN.
    """
    df = df.copy()
    if "temp_sydney" not in df.columns or df["temp_sydney"].isna().all():
        return df

    temp = df["temp_sydney"]
    BASE = 18.0

    # ---- Current temperature context (known at forecast time) ----
    df["temp_sydney"] = temp.astype(np.float32)
    df["hdd"]  = (BASE - temp).clip(lower=0).astype(np.float32)
    df["cdd"]  = (temp - BASE).clip(lower=0).astype(np.float32)
    # Quadratic terms capture non-linear demand response at extremes
    df["cdd_sq"] = (df["cdd"] ** 2).astype(np.float32)
    df["hdd_sq"] = (df["hdd"] ** 2).astype(np.float32)

    # Daily pattern
    df["temp_rmax_48"] = temp.rolling(48, min_periods=1).max().astype(np.float32)
    df["temp_rmin_48"] = temp.rolling(48, min_periods=1).min().astype(np.float32)
    df["temp_range_48"] = (df["temp_rmax_48"] - df["temp_rmin_48"]).astype(np.float32)

    # Regime flags
    df["heatwave_flag"] = (temp.rolling(96, min_periods=1).max() > 35).astype(np.float32)
    df["cold_flag"]     = (temp.rolling(48, min_periods=1).min() < 5).astype(np.float32)

    # Temperature change
    df["temp_chg_24h"] = temp.diff(48).astype(np.float32)

    # Historical lags for context
    for lag in [1, 2, 48, 96, 336]:
        df[f"temp_lag_{lag}"] = temp.shift(lag).astype(np.float32)

    # ---- 24h-ahead temperature (the key forward signal for next-day demand) ----
    # In training: actual temperature at T+24h (near-perfect proxy for BOM forecast)
    # In inference: replace with actual BOM 24h-ahead temperature forecast
    temp_t24 = temp.shift(-FORECAST_HORIZON)
    df["temp_t24"]    = temp_t24.astype(np.float32)
    df["hdd_t24"]     = (BASE - temp_t24).clip(lower=0).astype(np.float32)
    df["cdd_t24"]     = (temp_t24 - BASE).clip(lower=0).astype(np.float32)
    df["cdd_t24_sq"]  = (df["cdd_t24"] ** 2).astype(np.float32)
    df["hdd_t24_sq"]  = (df["hdd_t24"] ** 2).astype(np.float32)
    df["heatwave_t24"] = (temp_t24 > 35).astype(np.float32)
    df["cold_t24"]     = (temp_t24 < 5).astype(np.float32)
    df["temp_chg_t24_vs_now"] = (temp_t24 - temp).astype(np.float32)

    # ---- Newcastle temperature (Hunter Valley industrial load) ----
    if "temp_newcastle" in df.columns and not df["temp_newcastle"].isna().all():
        tn = df["temp_newcastle"]
        df["temp_newcastle"]   = tn.astype(np.float32)
        df["temp_nc_t24"]      = tn.shift(-FORECAST_HORIZON).astype(np.float32)
        df["temp_nc_cdd"]      = (tn - BASE).clip(lower=0).astype(np.float32)
        df["temp_nc_hdd"]      = (BASE - tn).clip(lower=0).astype(np.float32)
        df["temp_nc_cdd_t24"]  = ((tn.shift(-FORECAST_HORIZON) - BASE).clip(lower=0)).astype(np.float32)

    # ---- Feels-like temperature (thermal comfort proxy — better than dry temp) ----
    if "feelslike_sydney" in df.columns and not df["feelslike_sydney"].isna().all():
        fl = df["feelslike_sydney"]
        df["feelslike_cdd"]      = (fl - BASE).clip(lower=0).astype(np.float32)
        df["feelslike_hdd"]      = (BASE - fl).clip(lower=0).astype(np.float32)
        df["feelslike_cdd_sq"]   = (df["feelslike_cdd"] ** 2).astype(np.float32)
        fl_t24 = fl.shift(-FORECAST_HORIZON)
        df["feelslike_t24"]      = fl_t24.astype(np.float32)
        df["feelslike_cdd_t24"]  = (fl_t24 - BASE).clip(lower=0).astype(np.float32)
        df["feelslike_hdd_t24"]  = (BASE - fl_t24).clip(lower=0).astype(np.float32)
        df["feelslike_heatwave"] = (fl.rolling(96, min_periods=1).max() > 38).astype(np.float32)

    # ---- Humidity and dew point (amplify cooling/heating demand at extremes) ----
    if "humidity_sydney" in df.columns and not df["humidity_sydney"].isna().all():
        hu = df["humidity_sydney"]
        df["humidity_t24"]      = hu.shift(-FORECAST_HORIZON).astype(np.float32)
        df["humidity_rmean_48"] = hu.rolling(48, min_periods=1).mean().astype(np.float32)
        df["humid_high_flag"]   = (hu > 80).astype(np.float32)
        # Heat index proxy: cooling demand × humidity (amplification at high RH)
        if "feelslike_cdd" in df.columns:
            df["humidity_x_cdd"] = (df["feelslike_cdd"] * hu / 100.0).astype(np.float32)

    if "dew_sydney" in df.columns and not df["dew_sydney"].isna().all():
        dw = df["dew_sydney"]
        df["dew_t24"]         = dw.shift(-FORECAST_HORIZON).astype(np.float32)
        # High dew point (>18°C) → humid sultry conditions → more AC demand
        df["dew_high_flag"]   = (dw > 18).astype(np.float32)
        df["dew_t24_high"]    = (dw.shift(-FORECAST_HORIZON) > 18).astype(np.float32)

    # ---- Cross-region temperatures (neighboring state demand context) ----
    # QLD demand → QLD price → QNI interconnector pressure on NSW
    if "temp_brisbane" in df.columns and not df["temp_brisbane"].isna().all():
        tb = df["temp_brisbane"]
        df["cdd_brisbane"]      = (tb - BASE).clip(lower=0).astype(np.float32)
        df["hdd_brisbane"]      = (BASE - tb).clip(lower=0).astype(np.float32)
        df["temp_t24_brisbane"] = tb.shift(-FORECAST_HORIZON).astype(np.float32)
        df["cdd_t24_brisbane"]  = ((tb.shift(-FORECAST_HORIZON) - BASE).clip(lower=0)).astype(np.float32)

    # VIC demand → VIC price → VNI/Heywood interconnector pressure on NSW
    if "temp_melbourne" in df.columns and not df["temp_melbourne"].isna().all():
        tm = df["temp_melbourne"]
        df["cdd_melbourne"]      = (tm - BASE).clip(lower=0).astype(np.float32)
        df["hdd_melbourne"]      = (BASE - tm).clip(lower=0).astype(np.float32)
        df["temp_t24_melbourne"] = tm.shift(-FORECAST_HORIZON).astype(np.float32)
        df["cdd_t24_melbourne"]  = ((tm.shift(-FORECAST_HORIZON) - BASE).clip(lower=0)).astype(np.float32)

    # SA demand — SA-NSW path is indirect (via VIC) but useful in extreme events
    if "temp_adelaide" in df.columns and not df["temp_adelaide"].isna().all():
        ta = df["temp_adelaide"]
        df["cdd_adelaide"]      = (ta - BASE).clip(lower=0).astype(np.float32)
        df["hdd_adelaide"]      = (BASE - ta).clip(lower=0).astype(np.float32)
        df["temp_t24_adelaide"] = ta.shift(-FORECAST_HORIZON).astype(np.float32)

    return df

In [ ]:
def _add_solar_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Solar radiation features from local weather CSV files.

    Solar radiation is the dominant supply-side signal the codebase was
    previously missing:
      - High midday solar (>500 W/m²) suppresses net demand and spot price
      - The duck curve effect: price rises sharply in the evening as solar
        falls off and demand peaks from heating/cooking
      - A high-solar-tomorrow forecast reduces the incentive to generate
        thermally tonight, creating anticipatory price moves

    The 24h-ahead solar signal (shift(-FORECAST_HORIZON)) acts as a proxy
    for AEMO's own solar generation forecast at the target interval.
    At inference time, replace with actual BOM clear-sky / NWP forecast.

    No-op if solar_rad_sydney is absent or all-NaN.
    """
    df = df.copy()

    if "solar_rad_sydney" not in df.columns or df["solar_rad_sydney"].isna().all():
        return df

    sol = df["solar_rad_sydney"]

    # Compressed value — log1p since solar_rad >= 0, removes outlier skew
    df["solar_rad_log1p"]     = np.log1p(sol).astype(np.float32)

    # 24h-ahead solar (key forward signal: what will solar generation look like
    # at the actual forecast horizon?)
    sol_t24 = sol.shift(-FORECAST_HORIZON)
    df["solar_rad_t24"]       = sol_t24.astype(np.float32)
    df["solar_rad_t24_log1p"] = np.log1p(sol_t24).astype(np.float32)

    # Historical lags
    df["solar_rad_lag_48"]    = sol.shift(48).astype(np.float32)    # yesterday
    df["solar_rad_lag_96"]    = sol.shift(96).astype(np.float32)    # 2 days ago
    df["solar_rad_lag_336"]   = sol.shift(336).astype(np.float32)   # 1 week ago

    # Rolling stats (daytime solar context)
    df["solar_rad_rmean_48"]  = sol.rolling(48, min_periods=1).mean().astype(np.float32)
    df["solar_rad_rmax_48"]   = sol.rolling(48, min_periods=1).max().astype(np.float32)
    df["solar_rad_rmean_336"] = sol.rolling(336, min_periods=1).mean().astype(np.float32)

    # Daily cumulated solar proxy (sum of last 24 intervals ≈ 12h window)
    df["solar_cumday_24h"]    = sol.rolling(24, min_periods=1).sum().astype(np.float32)

    # Regime flags
    df["solar_high_flag"]     = (sol > 500).astype(np.float32)
    df["solar_t24_high_flag"] = (sol_t24 > 500).astype(np.float32)

    # Ramp: tomorrow's solar minus today's (rising solar day → price relief ahead)
    df["solar_ramp_t24"]      = (sol_t24 - sol).astype(np.float32)

    # Interaction: high forecast solar AND tight supply → conflict signal
    if "supply_stress" in df.columns:
        df["solar_t24_x_stress"] = (
            np.log1p(sol_t24) * df["supply_stress"].astype("float64")
        ).clip(-20, 20).astype(np.float32)

    # --- QLD (Brisbane) solar — affects QLD-NSW interconnector dispatch ---
    if "solar_rad_brisbane" in df.columns and not df["solar_rad_brisbane"].isna().all():
        sol_b = df["solar_rad_brisbane"]
        df["solar_rad_brisbane_t24"]      = sol_b.shift(-FORECAST_HORIZON).astype(np.float32)
        df["solar_rad_brisbane_rmean_48"] = sol_b.rolling(48, min_periods=1).mean().astype(np.float32)

    # --- VIC (Melbourne) solar — affects VIC-NSW interconnector dispatch ---
    if "solar_rad_melbourne" in df.columns and not df["solar_rad_melbourne"].isna().all():
        sol_m = df["solar_rad_melbourne"]
        df["solar_rad_melbourne_t24"] = sol_m.shift(-FORECAST_HORIZON).astype(np.float32)

    return df

In [ ]:
def _add_wind_cloud_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Wind speed and cloud cover features from local weather CSV files.

    Wind speed is a proxy for wind generation capacity:
      - NSW wind generation (mainly Capital and Sapphire wind farms) is modest
        but meaningful at the margin; Queensland wind is larger
      - High wind forecast → more renewable dispatch → lower thermal price

    Cloud cover affects solar generation and is the primary modulator of the
    solar radiation signal.

    No-op if windspeed_sydney is absent or all-NaN.
    """
    df = df.copy()

    # --- NSW wind speed ---
    if "windspeed_sydney" in df.columns and not df["windspeed_sydney"].isna().all():
        ws = df["windspeed_sydney"]

        df["windspeed_t24"]        = ws.shift(-FORECAST_HORIZON).astype(np.float32)
        df["windspeed_lag_48"]     = ws.shift(48).astype(np.float32)
        df["windspeed_rmean_48"]   = ws.rolling(48, min_periods=1).mean().astype(np.float32)
        df["windspeed_rmean_336"]  = ws.rolling(336, min_periods=1).mean().astype(np.float32)
        df["windspeed_high_flag"]  = (ws > 30).astype(np.float32)  # strong wind gen
        df["windspeed_t24_high"]   = (ws.shift(-FORECAST_HORIZON) > 30).astype(np.float32)

    # --- QLD wind speed ---
    if "windspeed_brisbane" in df.columns and not df["windspeed_brisbane"].isna().all():
        ws_b = df["windspeed_brisbane"]
        df["windspeed_brisbane_t24"]      = ws_b.shift(-FORECAST_HORIZON).astype(np.float32)
        df["windspeed_brisbane_rmean_48"] = ws_b.rolling(48, min_periods=1).mean().astype(np.float32)

    # --- NSW cloud cover ---
    if "cloudcover_sydney" in df.columns and not df["cloudcover_sydney"].isna().all():
        cc = df["cloudcover_sydney"]
        df["cloudcover_t24"]       = cc.shift(-FORECAST_HORIZON).astype(np.float32)
        df["cloudcover_lag_48"]    = cc.shift(48).astype(np.float32)
        df["cloudcover_rmean_48"]  = cc.rolling(48, min_periods=1).mean().astype(np.float32)
        df["clear_sky_flag"]       = (cc < 20).astype(np.float32)
        df["clear_sky_t24_flag"]   = (cc.shift(-FORECAST_HORIZON) < 20).astype(np.float32)
        df["overcast_t24_flag"]    = (cc.shift(-FORECAST_HORIZON) > 80).astype(np.float32)

        # Cloud-solar interaction: tomorrow cloud cover modulates solar forecast
        if "solar_rad_t24" in df.columns:
            cc_t24 = cc.shift(-FORECAST_HORIZON) / 100.0
            df["solar_x_cloud_t24"] = (
                df["solar_rad_t24"] * (1.0 - cc_t24.clip(0, 1))
            ).astype(np.float32)

    return df

In [ ]:
%reset -f